In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

jan_2026_dl_gen_ai_project_path = kagglehub.competition_download('jan-2026-dl-gen-ai-project')

print('Data source import complete.')


# Messy Mashup Genre Classification

## Milestone 1 — Dataset Exploration

This notebook explores the Messy Mashup dataset for the DL & Generative AI project.

Goals:
- Understand dataset structure
- Inspect audio stems
- Visualize spectrograms
- Analyze audio characteristics
- Create a baseline Kaggle submission

In [ ]:
import os
import numpy as np
import pandas as pd

import librosa
import librosa.display

import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
BASE = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup"

GENRES_PATH = os.path.join(BASE, "genres_stems")
MASHUPS_PATH = os.path.join(BASE, "mashups")
TEST_CSV = os.path.join(BASE, "test.csv")

In [ ]:
print(os.listdir(BASE))

In [ ]:
genres = sorted(os.listdir(GENRES_PATH))

print("Genres:", genres)
print("Total genres:", len(genres))

In [ ]:
sample_genre = genres[0]

genre_path = os.path.join(GENRES_PATH, sample_genre)

songs = os.listdir(genre_path)

print("Example songs:", songs[:5])

In [ ]:
song_folder = os.path.join(genre_path, songs[0])

print(os.listdir(song_folder))

In [ ]:
#Load one stem

drums_path = os.path.join(song_folder, "drums.wav")

audio, sr = librosa.load(drums_path, sr=22050)

print("Sample rate:", sr)
print("Audio length:", len(audio))
print("Duration (seconds):", len(audio)/sr)

In [ ]:
plt.figure(figsize=(10,4))

librosa.display.waveshow(audio, sr=sr)

plt.title("Drums Waveform")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")

plt.show()

In [ ]:
mel = librosa.feature.melspectrogram(
    y=audio,
    sr=sr,
    n_mels=128
)

mel_db = librosa.power_to_db(mel)

In [ ]:
plt.figure(figsize=(10,4))

librosa.display.specshow(
    mel_db,
    sr=sr,
    x_axis="time",
    y_axis="mel"
)

plt.title("Mel Spectrogram")
plt.colorbar()

plt.show()

In [ ]:
# Combine Stems to Simulate Full Song

stems = ["drums.wav","bass.wav","vocals.wav","other.wav"]

audio_sum = None
sr = 22050

for stem in stems:

    path = os.path.join(song_folder, stem)

    if not os.path.exists(path):
        continue

    try:
        y, sr = librosa.load(path, sr=sr)
    except:
        continue

    if audio_sum is None:
        audio_sum = y
    else:
        min_len = min(len(audio_sum), len(y))
        audio_sum = audio_sum[:min_len] + y[:min_len] * 0.25

if audio_sum is None:
    audio_sum = np.zeros(sr*5)

print("Combined audio samples:", len(audio_sum))
print("Duration (seconds):", len(audio_sum)/sr)


In [ ]:
test_df = pd.read_csv(TEST_CSV)

test_df.head()

In [ ]:
sample_mashup = os.path.join(MASHUPS_PATH, "song0001.wav")

audio, sr = librosa.load(sample_mashup, sr=22050)

plt.figure(figsize=(10,4))

librosa.display.waveshow(audio, sr=sr)

plt.title("Mashup Audio")
plt.show()

In [ ]:
# Baseline submission... random prediction

import random

predictions = []

for _ in range(len(test_df)):
    predictions.append(random.choice(genres))

In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()